# Pooled scenario metrics generation

Reconstruct the released **MISO_NCA** North/Central pool from five MISO
subregions for 2007–2023. A **scenario** is a wind/solar capacity mix.

Read member generation → verify matching hours → sum generation and capacity
→ calculate weighted pooled CF → apply six scenarios → compare and save.

**Inputs:** installed-scenario generation for the five member BAs, bundled BA
capacity metadata, and the complete released MISO_NCA scenario table for comparison.

**Requirements:** the repository environment and bundled inputs;
no weather-service access or model training is needed.

**Outputs:** one reconstructed CSV under
`data_outputs/data_flow/pooled_scenario_metrics_generation/`, plus inline member
contributions, capacity tables, and archive-comparison results. The released files
are read as inputs and remain unchanged.

Run the code cells from top to bottom. See the [setup instructions](../../README.md#quick-start).

[BA scenario generation](ba_scenario_metrics_generation.ipynb) explains the member
ingredients. MISO_NCA contains five North/Central subregions; it excludes the South
subregion `MISO_8910`. It is distinct from direct `MISO` and the six-member
`MISO_SUBREGION_SUM`. The [pool membership table](../../README.md#coverage) documents
the other released groups.

## Setup

Keep the pool membership, input locations, and displayed example hour together.
The calculation below is specific to these five members, which all have load,
wind, solar, and capacity metadata for all six scenarios.

The version `0.1.0` example inputs are bundled under `data_inputs/examples/`;
no archive extraction is needed. For another pool, obtain its inputs from the
full archive and update the membership and paths.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

POOL_CODE = "MISO_NCA"
MEMBERS = ["MISO_0001", "MISO_0004", "MISO_0006", "MISO_0027", "MISO_0035"]
DATASET = "wtk_bchrrr_nsrdb_2007_2023"
DATA_DIR = Path("../../data_inputs/examples") / DATASET
OUTPUT_DIR = Path("../../data_outputs/data_flow/pooled_scenario_metrics_generation")
EXAMPLE_HOUR = pd.Timestamp("2021-02-17T18:00:00Z")

SCENARIO_LABELS = {
    "installed_2024": "Installed 2024",
    "split_w00_s100": "0% wind / 100% solar",
    "split_w25_s75": "25% wind / 75% solar",
    "split_w50_s50": "50% wind / 50% solar",
    "split_w75_s25": "75% wind / 25% solar",
    "split_w100_s00": "100% wind / 0% solar",
}
GENERATION_COLUMNS = ["load_mw", "wind_mw", "solar_mw"]
CAPACITY_COLUMNS = ["wind_capacity_mw", "solar_capacity_mw"]

## Step 1: Read existing BA products

The member files already contain modeled load, wind generation, and solar
generation in **MW**. Read the `installed_2024` rows directly; do not recalculate
generation from raw weather or CF. Their generation already reflects the source
loss assumptions. The bundled member tables retain only these installed-scenario
rows and the five required columns.

In [2]:
member_inputs = {}
for member in MEMBERS:
    path = DATA_DIR / "ba_scenario_metrics" / f"{member}_{DATASET}_scenario_metrics.csv.gz"
    member_generation = pd.read_csv(path, usecols=["time_utc", "scenario", *GENERATION_COLUMNS])
    member_generation = member_generation.loc[member_generation["scenario"].eq("installed_2024")].copy()
    member_generation["time_utc"] = pd.to_datetime(member_generation["time_utc"], utc=True)
    member_inputs[member] = member_generation

print(f"Read installed-scenario generation for all {len(MEMBERS)} members of {POOL_CODE}.")

Read installed-scenario generation for all 5 members of MISO_NCA.


The metadata repeats capacities across metric rows. Keep one capacity record
per member and scenario so each capacity is counted once.

In [3]:
metadata = pd.read_csv("../../manifests/ba_scenario_metric_metadata_manifest_wtk_bchrrr_nsrdb_2007_2023.csv", usecols=["ba_code", "scenario", *CAPACITY_COLUMNS])
member_capacities = metadata.loc[metadata["ba_code"].isin(MEMBERS)].copy()
member_capacities = member_capacities.drop_duplicates()

assert not member_capacities.duplicated(["ba_code", "scenario"]).any(), ("Conflicting capacity records for a member and scenario.")
assert np.isfinite(member_capacities[CAPACITY_COLUMNS].to_numpy(dtype=float)).all(), ("Capacity metadata must contain finite values.")
assert member_capacities[CAPACITY_COLUMNS].ge(0).all().all(), "Capacity cannot be negative."
for member in MEMBERS:
    member_scenarios = member_capacities.loc[member_capacities["ba_code"].eq(member)]
    assert set(member_scenarios["scenario"]) == set(SCENARIO_LABELS), (f"Expected all six capacity scenarios for {member}.")

print(f"Read {len(member_capacities)} unique capacity records: five members x six scenarios.")

Read 30 unique capacity records: five members x six scenarios.


## Step 2: Verify synchronization

The released member products already omit February 29. Each must contain the
same **148,920 UTC hours**, with no duplicate timestamps or missing values.
The expected calendar below is used only for checking: no input rows are dropped,
filled, or interpolated. Sorting makes the comparison independent of file order.

In [4]:
# Construct the expected no-leap calendar without modifying any member input.
expected_times = pd.date_range("2007-01-01", "2023-12-31 23:00", freq="h", tz="UTC", name="time_utc")
expected_leap_days = (expected_times.month == 2) & (expected_times.day == 29)
expected_times = expected_times[~expected_leap_days]

member_hourly = {}
coverage_rows = []
for member in MEMBERS:
    member_generation = member_inputs[member].sort_values("time_utc")
    member_generation = member_generation.set_index("time_utc")
    pd.testing.assert_index_equal(member_generation.index, expected_times, check_names=False, obj=f"{member} UTC calendar")
    assert np.isfinite(member_generation[GENERATION_COLUMNS].to_numpy(dtype=float)).all(), (f"Missing or invalid hourly values in {member}.")
    member_hourly[member] = member_generation[GENERATION_COLUMNS]
    coverage_rows.append({"Member": member, "Verified hours": len(member_generation)})

print(f"Identical calendars: {expected_times.min()} through {expected_times.max()}.")
display(pd.DataFrame(coverage_rows).set_index("Member"))

Identical calendars: 2007-01-01 00:00:00+00:00 through 2023-12-31 23:00:00+00:00.


,Verified hours
Member,
MISO_0001,148920
MISO_0004,148920
MISO_0006,148920
MISO_0027,148920
MISO_0035,148920


## Step 3: Sum installed ingredients

Add the members' load and generation at the **same hour**:

$$L_{\mathrm{pool},t}=\sum_i L_{i,t},\qquad
G_{w,\mathrm{pool},t}=\sum_i G_{w,i,t},\qquad
G_{s,\mathrm{pool},t}=\sum_i G_{s,i,t}.$$

Installed capacity is added separately for wind and solar. These are nameplate
MW, while the hourly columns are modeled output in MW. Only displayed values
are rounded; the calculations keep full precision.

In [5]:
installed_capacities = member_capacities.loc[member_capacities["scenario"].eq("installed_2024")]
installed_capacities = installed_capacities.set_index("ba_code").loc[MEMBERS]
installed_wind_capacity_mw = installed_capacities["wind_capacity_mw"].sum()
installed_solar_capacity_mw = installed_capacities["solar_capacity_mw"].sum()
installed_total_capacity_mw = installed_wind_capacity_mw + installed_solar_capacity_mw
assert installed_wind_capacity_mw > 0 and installed_solar_capacity_mw > 0

In [6]:
pooled_installed = pd.DataFrame(0.0, index=expected_times, columns=GENERATION_COLUMNS)
for member in MEMBERS:
    pooled_installed = pooled_installed + member_hourly[member]

**Display: Member contributions and pool totals at the example hour.**

In [7]:
contribution_rows = []
for member in MEMBERS:
    example_generation = member_hourly[member].loc[EXAMPLE_HOUR]
    member_capacity = installed_capacities.loc[member]
    contribution_rows.append(
        {
            "Member / pool": member,
            "Load (MW)": example_generation["load_mw"],
            "Wind generation (MW)": example_generation["wind_mw"],
            "Wind capacity (MW)": member_capacity["wind_capacity_mw"],
            "Solar generation (MW)": example_generation["solar_mw"],
            "Solar capacity (MW)": member_capacity["solar_capacity_mw"],
        }
    )

pooled_example_generation = pooled_installed.loc[EXAMPLE_HOUR]
contribution_rows.append(
    {
        "Member / pool": POOL_CODE,
        "Load (MW)": pooled_example_generation["load_mw"],
        "Wind generation (MW)": pooled_example_generation["wind_mw"],
        "Wind capacity (MW)": installed_wind_capacity_mw,
        "Solar generation (MW)": pooled_example_generation["solar_mw"],
        "Solar capacity (MW)": installed_solar_capacity_mw,
    }
)
contributions = pd.DataFrame(contribution_rows).set_index("Member / pool")
print(f"Installed-scenario contributions at {EXAMPLE_HOUR}:")
display(contributions.round(1))

Installed-scenario contributions at 2021-02-17 18:00:00+00:00:


,Load (MW),Wind generation (MW),Wind capacity (MW),Solar generation (MW),Solar capacity (MW)
Member / pool,,,,,
MISO_0001,12719.9,1393.1,9732.7,868.1,1647.6
MISO_0004,6659.9,5.8,2768.8,1339.4,2467.1
MISO_0006,12658.6,0.0,1441.5,749.4,1348.6
MISO_0027,20917.6,84.2,4603.7,1679.5,3248.9
MISO_0035,13071.7,1070.9,13419.5,632.5,1069.0
MISO_NCA,66027.8,2553.9,31966.2,5269.0,9781.2


<a id="step-4-calculate-weighted-pooled-cf-and-combine-portfolio-capacities"></a>

## Step 4: Calculate weighted pooled CF

Divide installed generation (MW) by installed capacity (MW) for each technology
to obtain a dimensionless CF:

$$CF_{w,\mathrm{pool},t}=\frac{\sum_i G_{w,i,t}}{\sum_i C_{w,i}}
=\frac{\sum_i CF_{w,i,t} C_{w,i}}{\sum_i C_{w,i}}.$$

Solar follows the same equation. This is capacity weighting, not an equal average
of member CFs. It preserves the installed geographic distribution represented by
each technology. Losses already present in member generation are not applied again.

In [8]:
pooled_profiles = pooled_installed[["load_mw"]].copy()
pooled_profiles["wind_cf"] = pooled_installed["wind_mw"] / installed_wind_capacity_mw
pooled_profiles["solar_cf"] = pooled_installed["solar_mw"] / installed_solar_capacity_mw

**Display: Wind and solar CF divisions at the example hour.**

In [9]:
example_wind_mw = pooled_installed.loc[EXAMPLE_HOUR, "wind_mw"]
example_solar_mw = pooled_installed.loc[EXAMPLE_HOUR, "solar_mw"]
example_wind_cf = pooled_profiles.loc[EXAMPLE_HOUR, "wind_cf"]
example_solar_cf = pooled_profiles.loc[EXAMPLE_HOUR, "solar_cf"]

print(f"Pooled capacity factors at {EXAMPLE_HOUR}:")
print(f'Wind CF: {example_wind_mw:,.1f} MW / {installed_wind_capacity_mw:,.1f} MW = {example_wind_cf:.3f} ({example_wind_cf:.1%})')
print(f'Solar CF: {example_solar_mw:,.1f} MW / {installed_solar_capacity_mw:,.1f} MW = {example_solar_cf:.3f} ({example_solar_cf:.1%})')

Pooled capacity factors at 2021-02-17 18:00:00+00:00:
Wind CF: 2,553.9 MW / 31,966.2 MW = 0.080 (8.0%)
Solar CF: 5,269.0 MW / 9,781.2 MW = 0.539 (53.9%)


## Step 5: Calculate pooled scenario metrics

Sum each scenario's member capacities, then apply them to the pooled CF profiles:

$$G_w=CF_w C_w,\qquad G_s=CF_s C_s,$$
$$L_{\mathrm{net}}=L-G_w-G_s,\qquad
CF_{\mathrm{equiv}}=\frac{G_w+G_s}{C_w+C_s}.$$

Load and technology-specific CF stay the same across scenarios. Capacity,
generation, net load, and equivalent renewable CF depend on the selected mix.

All five members have the same six scenarios. Sum their existing capacities from
the metadata instead of recreating the percentage splits. The installed scenario
keeps the reference fleet's wind/solar mix; the five synthetic scenarios change
those capacity shares while holding total renewable nameplate capacity fixed.
These are capacity shares, not hourly energy shares.

**Sum the scenario capacities, not the member synthetic generation.** Giving
each member a 50/50 mix would change the geographic weights. This pooled product
uses the installed-weighted CF profiles for the whole region.

In [10]:
scenario_capacities = member_capacities.groupby("scenario")[CAPACITY_COLUMNS].sum()
scenario_capacities = scenario_capacities.loc[list(SCENARIO_LABELS)].copy()
scenario_capacities["total_capacity_mw"] = (scenario_capacities["wind_capacity_mw"] + scenario_capacities["solar_capacity_mw"])
# Published metadata round capacities; allow the same tolerance as the archive comparison.
np.testing.assert_allclose(scenario_capacities["total_capacity_mw"], installed_total_capacity_mw, rtol=1e-5, atol=1e-5)

**Display: Wind and solar capacities for the six pooled scenarios.**

In [11]:
scenario_capacity_preview = scenario_capacities.rename(index=SCENARIO_LABELS)
scenario_capacity_preview = scenario_capacity_preview.rename_axis("Scenario")
scenario_capacity_preview = scenario_capacity_preview.rename(
    columns={
        "wind_capacity_mw": "Wind capacity (MW)",
        "solar_capacity_mw": "Solar capacity (MW)",
        "total_capacity_mw": "Total capacity (MW)",
    }
)
display(scenario_capacity_preview.round(1))

,Wind capacity (MW),Solar capacity (MW),Total capacity (MW)
Scenario,,,
Installed 2024,31966.2,9781.2,41747.4
0% wind / 100% solar,0.0,41747.4,41747.4
25% wind / 75% solar,10436.8,31310.5,41747.4
50% wind / 50% solar,20873.7,20873.7,41747.4
75% wind / 25% solar,31310.5,10436.8,41747.4
100% wind / 0% solar,41747.4,0.0,41747.4


In [12]:
scenario_tables = []
for scenario, capacities in scenario_capacities.iterrows():
    hourly = pooled_profiles.copy()
    hourly["scenario"] = scenario
    hourly["wind_mw"] = hourly["wind_cf"] * capacities["wind_capacity_mw"]
    hourly["solar_mw"] = hourly["solar_cf"] * capacities["solar_capacity_mw"]
    renewable_mw = hourly["wind_mw"] + hourly["solar_mw"]
    hourly["net_load_mw"] = hourly["load_mw"] - renewable_mw
    hourly["renew_cf_equiv"] = renewable_mw / capacities["total_capacity_mw"]
    scenario_tables.append(hourly)

**Assemble:** Stack the six scenarios in the released column order.

In [13]:
SCENARIO_METRIC_COLUMNS = [
    "time_utc",
    "scenario",
    "load_mw",
    "wind_cf",
    "solar_cf",
    "wind_mw",
    "solar_mw",
    "net_load_mw",
    "renew_cf_equiv",
]
notebook_scenarios = pd.concat(scenario_tables)
notebook_scenarios = notebook_scenarios.reset_index()
notebook_scenarios = notebook_scenarios[SCENARIO_METRIC_COLUMNS]

**Display: Compare the six scenarios at the same example hour.**

In [14]:
scenario_preview = notebook_scenarios.loc[notebook_scenarios["time_utc"].eq(EXAMPLE_HOUR), ["scenario", "wind_mw", "solar_mw", "net_load_mw", "renew_cf_equiv"]]
scenario_preview = scenario_preview.set_index("scenario").rename(index=SCENARIO_LABELS)
scenario_preview = scenario_preview.rename_axis("Scenario")
scenario_preview = scenario_preview.rename(
    columns={
        "wind_mw": "Wind generation (MW)",
        "solar_mw": "Solar generation (MW)",
        "net_load_mw": "Net load (MW)",
        "renew_cf_equiv": "Renewable-equivalent CF (fraction)",
    }
)
print(f"Pooled scenarios at {EXAMPLE_HOUR}:")
display(
    scenario_preview.round(
        {
            "Wind generation (MW)": 1,
            "Solar generation (MW)": 1,
            "Net load (MW)": 1,
            "Renewable-equivalent CF (fraction)": 3,
        }
    )
)

Pooled scenarios at 2021-02-17 18:00:00+00:00:


,Wind generation (MW),Solar generation (MW),Net load (MW),Renewable-equivalent CF (fraction)
Scenario,,,,
Installed 2024,2553.9,5269.0,58204.8,0.187
0% wind / 100% solar,0.0,22489.0,43538.8,0.539
25% wind / 75% solar,833.8,16866.7,48327.2,0.424
50% wind / 50% solar,1667.7,11244.5,53115.6,0.309
75% wind / 25% solar,2501.5,5622.2,57904.0,0.195
100% wind / 0% solar,3335.4,0.0,62692.4,0.080


## Step 6: Compare with the released product and save

Check the full **893,520-row** reconstruction: six scenarios with 148,920 hours
each. Sort by scenario and timestamp before comparing every released column.
Use `rtol=1e-5` and `atol=1e-5` for the numerical comparison.
Rounded capacity metadata can create small numerical differences;
the table reports the maximum absolute difference for each metric.

Write a notebook-generated CSV only after the comparison passes. This numerical
check does not require identical CSV bytes and is separate from observational
validation of the underlying load and renewable models.

In [15]:
archive_path = DATA_DIR / "pooled_scenario_metrics" / f"{POOL_CODE}_{DATASET}_scenario_metrics.csv.gz"
archive = pd.read_csv(archive_path, parse_dates=["time_utc"])
sort_columns = ["scenario", "time_utc"]
reconstructed_sorted = notebook_scenarios.sort_values(sort_columns).reset_index(drop=True)
released_sorted = archive.sort_values(sort_columns).reset_index(drop=True)
pd.testing.assert_frame_equal(reconstructed_sorted, released_sorted, check_dtype=False, rtol=1e-5, atol=1e-5)

metric_columns = SCENARIO_METRIC_COLUMNS[2:]
metric_differences = reconstructed_sorted[metric_columns] - released_sorted[metric_columns]
absolute_differences = metric_differences.abs()
max_abs_diff = absolute_differences.max()

**Display: Maximum differences from the release, in MW or dimensionless CF.**

In [16]:
comparison_metrics = [
    ("load_mw", "Load", "MW"),
    ("wind_cf", "Wind capacity factor", "CF (fraction)"),
    ("solar_cf", "Solar capacity factor", "CF (fraction)"),
    ("wind_mw", "Wind generation", "MW"),
    ("solar_mw", "Solar generation", "MW"),
    ("net_load_mw", "Net load", "MW"),
    ("renew_cf_equiv", "Renewable-equivalent capacity factor", "CF (fraction)"),
]
comparison_rows = []
for column, label, unit in comparison_metrics:
    comparison_rows.append(
        {
            "Metric": label,
            "Units": unit,
            "Maximum absolute difference": max_abs_diff[column],
        }
    )
comparison = pd.DataFrame(comparison_rows).set_index("Metric")
print(f"Archive comparison passed for all {len(reconstructed_sorted):,} {POOL_CODE} scenario rows.")
display(comparison.style.format({"Maximum absolute difference": "{:.6e}"}))

Archive comparison passed for all 893,520 MISO_NCA scenario rows.


,Units,Maximum absolute difference
Metric,,
Load,MW,1.455192e-11
Wind capacity factor,CF (fraction),3.474232e-11
Solar capacity factor,CF (fraction),1.246072e-10
Wind generation,MW,1.973014e-06
Solar generation,MW,4.812016e-06
Net load,MW,4.812020e-06
Renewable-equivalent capacity factor,CF (fraction),1.246072e-10


**Save:** Write the hourly values that passed the archive comparison.

In [17]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_csv = OUTPUT_DIR / f"{POOL_CODE}_{DATASET}_notebook_scenario_metrics.csv"
notebook_scenarios.to_csv(output_csv, index=False, date_format="%Y-%m-%dT%H:%M:%SZ")
print(f"Wrote reconstructed scenario metrics: {output_csv.as_posix()}")

Wrote reconstructed scenario metrics: ../../data_outputs/data_flow/pooled_scenario_metrics_generation/MISO_NCA_wtk_bchrrr_nsrdb_2007_2023_notebook_scenario_metrics.csv


## Interpretation

This reconstruction checks the complete released MISO_NCA product within the
stated tolerance. The wind and solar CF profiles represent a **fixed 2024
operable reference fleet under 2007–2023 weather**, rather than the changing
fleet that actually operated in each historical year. Capacity weighting gives
the aggregate modeled CF of that reference fleet; it does not remove errors in
the member models or establish agreement with observed generation.

Each hypothetical scenario preserves the installed geographic distribution
within wind and within solar while changing their combined capacity shares.
The member-capacity shortcut works here because all five members have all six
scenarios; it should not be assumed to work for other membership lists.

Pooling adds synchronous load and generation without transmission limits or
dispatch constraints. It does not demonstrate that all modeled generation can
be delivered to every location. MISO_NCA also must not be counted again alongside
its constituent subregions or overlapping MISO aggregates.

Next, [stress-event detection](ba_stress_event_catalog.ipynb) shows how hourly
scenario metrics become event catalogs. [Pairwise pooling analysis](../analysis/pairwise_pooling_heatmap.ipynb)
uses scenario metrics to study threshold-relative resource sharing. Those
notebooks retain their stated example regions; their MISO subregion-sum product
contains all six subregions, unlike this five-member North/Central example.